# Tutorial 1 -- Multiphoton STAR Bloch Simulations

This tutorial follows the complete workflow with simulation and pulse parameters suitable to run on a local computer. First, the one-photon reference pulse is constructed. With this reference, constant- and phase-modulated-$B_{1,z}$ two-photon pulses are generated with $n=-1$. Each pulse sequence is simulated for a Shepp-Logan phantom. After simulation, multiphoton images are reconstructed from the $m=+1$ replicas following the process detailed in the manuscript.

In [ ]:
from pathlib import Path
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src' / 'multiphoton_star').is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError('Run this notebook from within the multiphoton-star-toolbox repository.')
sys.path.insert(0, str(repo_root / 'src'))

import matplotlib.pyplot as plt
import numpy as np
from multiphoton_star import phantoms, plotting, pulse, radial, recon, simulate

## 1. Pulse sequence and simulation parameters

The pulse uses the paper's $\beta=5.3$ and HS shape order $q=2$. Spatial resolution, output matrix, gradient extent, and spoke count are derived by `generate_radial_gradients` from the time-bandwidth product $R$, bandwidth, pulse duration, and FOV. The resulting Bloch simulation can be computationally expensive depending on these parameters; parallelization is supported.

Important considerations for pulse and simulation parameters:
- The time-bandwidth product $R$ strongly affects image quality and sets the spatial resolution together with the bandwidth and FOV. Larger $R$ also reduces finite-pulse effects and makes the deconvolution better posed.
- Simulation cost grows roughly with $R^2$ (both the spoke count and the samples per spoke scale with $R$), with the object size, and with the number of time samples per $f_z$ period. With the default settings ($R=200$, $f_z = 3\,$ `bw_hz`, about 11 samples per $f_z$ period, a 200 x 200 object), each sequence takes roughly 35 minutes with one worker and about 2.5 GB of memory per worker; larger $R$, larger objects, or larger $f_z$ may require a cluster.
- The simulation object should be larger than the final image matrix so that it approximates a continuous object.
- The simulator time step `dt_s` should be small relative to the fastest variation in any simulator input; for multiphoton simulations, this is typically the $B_{1,z}$ oscillation at $f_z$.
- The manuscript uses $f_z \gg$ `bw_hz` so that the model assumptions are well satisfied, but smaller offsets can still give reasonable images: $f_z \geq$ `bw_hz` can be sufficient for constant $B_{1,z}$, and $f_z \geq 2.5\,$ `bw_hz` generally gives reasonable error for phase-modulated $B_{1,z}$. Larger $f_z$ relative to `bw_hz` further improves the reconstruction, but requires a smaller `dt_s` and therefore more computation.

In [ ]:
beta = 5.3
q = 2               

bw_hz = 10_000.0    # Bandwidth
R = 200.0           # Time-bandwidth product
dt_s = 1.0 / (11 * 3 * bw_hz)  # Simulator time step: ~11 samples per f_z period
B1xy_max_G = 5e-3   # Can be varied to change flip angle.
f_z_hz = 3 * bw_hz  # Multiphoton frequency offset (f_z/bw = 3)
n_workers = 3       # Parallel simulation workers (~2.5 GB memory each)
FOV_cm = (4.0, 4.0) # Desired FOV

# Simulation object size (should approximate continuous, real object; must be sufficiently larger than simulated output matrix size)
phantom_Ny = 200
phantom_Nx = 200

samples_per_f_z_period = 1.0 / (f_z_hz * dt_s)
print(f'Samples per f_z period: {samples_per_f_z_period:.1f}')

## 2. Construct and inspect the one-photon pulse
The number of time points/samples that will be simulated for each TR is printed. This can be used to get an idea of how computationally complex the time and pulse parameters will be.

In [ ]:
# Generate target pulse
# One-photon reference pulse used to construct the multiphoton waveforms.
reference = pulse.generate_hs_pulse(
    beta=beta, q=q, bw_hz=bw_hz, R=R, dt_s=dt_s,
    B1xy_max_G=B1xy_max_G, verbose=False,
)
print(f'{reference.M} samples, {1e3 * reference.duration_s:.1f} ms duration')
print(f'Estimated one-photon flip angle: {reference.flip_angle_deg:.2f} degrees')

In [ ]:
figure, _ = plotting.plot_waveforms(
    reference.t_s, reference.B1xy_G, domain='both', normalize_spectra=True
);
figure.suptitle('One-photon HS pulse');

## 3. Construct two-photon transmit waveforms

Here, the reference pulse is converted into the $B_{1,xy}$ and $B_{1,z}$ fields for the two two-photon sequences ($n=-1$). The $B_{1,xy}$ carrier is shifted in frequency by $nf_z$. The constant-$B_{1,z}$ sequence keeps the HS sweep in $B_{1,xy}$, while the phase-modulated-$B_{1,z}$ sequence moves the sweep into the phase of $B_{1,z}$. The $B_{1,z}$ amplitude is set by the modulation index $a=\gamma B_{1,z}/\omega_z$ at the first maximum of $|J_n(a)|$, and $B_{1,xy}$ is scaled by $1/|J_n(a)|$, so the two-photon pulses have the same flip angle as the one-photon reference (small-flip approximation).

In [ ]:
# Constant-B1z two-photon pulse
constant_fields = reference.make_multiphoton(
    n=-1, f_z_Hz=f_z_hz, mode='constant', verbose=False
)

# Phase-Modulated-B1z two-photon pulse
phase_modulated_fields = reference.make_multiphoton(
    n=-1, f_z_Hz=f_z_hz, mode='phase_modulated', verbose=False
)
print(f'a = {constant_fields.a:.6f}')
print(f'J_n(a) = {constant_fields.excitation_coefficient:.6f}')
print(f'B1xy scale = {constant_fields.B1xy_scale:.6f}')

In [ ]:
# Since the oscillations from f_z are fast, typically only the envelope is visible in the time-domain view. Frequency domain view is more intuitive.
figure, _ = plotting.plot_waveforms(
    reference.t_s, constant_fields.B1xy_G, constant_fields.B1z_G,
    domain='both', normalize_spectra=True,
);
figure.suptitle(r'Constant-$B_{1,z}$ two-photon fields ($n=-1$)');

In [ ]:
figure, _ = plotting.plot_waveforms(
    reference.t_s, phase_modulated_fields.B1xy_G, phase_modulated_fields.B1z_G,
    domain='both', normalize_spectra=True,
);
figure.suptitle(r'Phase-modulated-$B_{1,z}$ two-photon fields ($n=-1$)');

## 4. Build simulation object and gradients
The simulation object size and the number of radial spokes (equal to the number of TRs) strongly affect computational cost. The simulation object size does not set the final resolution. However, if the object is not at least 1.5 times (ideally 2 times) larger than the final image size, the simulated signal approximates a continuous object less accurately, and the check below stops the notebook. The check can be removed if desired.

By default, the gradient is chosen so that the FOV spans half the excitation bandwidth, and the reconstruction size follows from the resulting spatial resolution. The number of radial spokes is derived from the spatial resolution to meet Nyquist.

In [ ]:
# k-space trajectory
trajectory = radial.generate_radial_gradients(
    bw_hz=bw_hz, FOV_cm=FOV_cm, pulse=reference,
    verbose=False,
)
resolution_cm = trajectory.resolution_cm
reconstruction_Nx = int(round(FOV_cm[0] / resolution_cm))
reconstruction_Ny = int(round(FOV_cm[1] / resolution_cm))

# Generate Shepp-Logan phantom simulation object
obj = phantoms.shepp_logan_bloch(
    Ny=phantom_Ny, Nx=phantom_Nx, FOV_cm=FOV_cm,
    threshold=1e-6,
)
expected_spokes = int(np.ceil(np.pi * min(FOV_cm) / resolution_cm))
plt.figure(figsize=(4, 4))
plt.imshow(obj.img, cmap='gray', origin='lower', extent=(-2, 2, -2, 2))
plt.xlabel('x (cm)')
plt.ylabel('y (cm)')
plt.title(f'{phantom_Ny}x{phantom_Nx} simulation object');
print(f'Simulation object: {phantom_Ny}x{phantom_Nx}')
print(f'Reconstruction: {reconstruction_Ny}x{reconstruction_Nx}')
print(f'{trajectory.N_spokes} radial spokes')
print(f'Trajectory resolution: {trajectory.resolution_cm:.3f} cm')

# Checks
assert phantom_Nx >= 1.5 * reconstruction_Nx
assert phantom_Ny >= 1.5 * reconstruction_Ny
assert trajectory.N_spokes == expected_spokes

## 5. Run the Bloch simulator

Each cell runs a Bloch simulation for each pulse sequence with the same gradient trajectory and simulation object.

In [ ]:
%%time
# One-photon reference sequence
one_photon_sim = simulate.simulate_radial_bloch(
    pulse=reference, traj=trajectory, positions_cm=obj.positions_cm,
    mx0=obj.mx0, my0=obj.my0, mz0=obj.mz0,
    workers=n_workers, progress=True, tqdm_desc='One photon',
)

In [ ]:
%%time
# Two-photon, constant-B1z sequence
constant_b1z_sim = simulate.simulate_radial_bloch(
    pulse=reference, traj=trajectory, positions_cm=obj.positions_cm,
    mx0=obj.mx0, my0=obj.my0, mz0=obj.mz0,
    workers=n_workers, progress=True, tqdm_desc='Constant B1z',
    **constant_fields.to_sim_kwargs(),
)

In [ ]:
%%time
# Two-photon, phase-modulated-B1z sequence
phase_modulated_b1z_sim = simulate.simulate_radial_bloch(
    pulse=reference, traj=trajectory, positions_cm=obj.positions_cm,
    mx0=obj.mx0, my0=obj.my0, mz0=obj.mz0,
    workers=n_workers, progress=True, tqdm_desc='Phase-modulated B1z',
    **phase_modulated_fields.to_sim_kwargs(),
)

### Display single spokes
Two-photon raw spokes generally have a similar magnitude envelope as the one-photon case, but many additional oscillations. These are better appreciated in the frequency-domain, where the replica structure is visible.

In [ ]:
spoke_index = 0

one_photon_figure, _ = plotting.plot_spoke(
    one_photon_sim, spoke_index=spoke_index,
)
one_photon_figure.suptitle(f'One photon: raw spoke {spoke_index}')

constant_figure, _ = plotting.plot_spoke(
    constant_b1z_sim, spoke_index=spoke_index,
)
constant_figure.suptitle(
    rf'Constant-$B_{{1,z}}$: raw spoke {spoke_index}'
)

phase_modulated_figure, _ = plotting.plot_spoke(
    phase_modulated_b1z_sim, spoke_index=spoke_index,
)
phase_modulated_figure.suptitle(
    rf'Phase-modulated-$B_{{1,z}}$: raw spoke {spoke_index}'
)

## 6. Reconstruct Images

The multiphoton reconstruction obtains $n$, $a$, the transverse scale, and the longitudinal mode from the generated fields. We choose only the received replica index $m$. This can be changed to any integer value. 

In [ ]:
# 1. Deconvolve spokes

one_photon_spokes = recon.deconvolve_with_pulse(one_photon_sim.raw_mat, reference)
constant_b1z_spokes = recon.deconvolve_replica(
    raw_mat=constant_b1z_sim.raw_mat, pulse=reference, waveforms=constant_fields,
    m=+1, bw_Hz=bw_hz,
)
phase_modulated_b1z_spokes = recon.deconvolve_replica(
    raw_mat=phase_modulated_b1z_sim.raw_mat, pulse=reference,
    waveforms=phase_modulated_fields, m=+1, bw_Hz=bw_hz,
)

In [ ]:
# 2. Radial image recon

one_photon_image = recon.reconstruct_radial_image_2d(
    one_photon_spokes.x_time, trajectory, FOV_cm=FOV_cm,
    dcf='ramp', edge_taper_r0=0.85, edge_taper_kind='tukey',
    eps=1e-6, grid_oversamp=2.0,
)
constant_b1z_image = recon.reconstruct_radial_image_2d(
    constant_b1z_spokes.x_time, trajectory, FOV_cm=FOV_cm,
    dcf='ramp', edge_taper_r0=0.85, edge_taper_kind='tukey',
    eps=1e-6, grid_oversamp=2.0,
)
phase_modulated_b1z_image = recon.reconstruct_radial_image_2d(
    phase_modulated_b1z_spokes.x_time, trajectory, FOV_cm=FOV_cm,
    dcf='ramp', edge_taper_r0=0.85, edge_taper_kind='tukey',
    eps=1e-6, grid_oversamp=2.0,
)

## 7. Inspect and compare the reconstructed images

In [ ]:
images = {
    'One photon': one_photon_image.F_cart,
    r'Constant $B_{1,z}$': constant_b1z_image.F_cart,
    r'Phase-modulated $B_{1,z}$': phase_modulated_b1z_image.F_cart,
}
reference_array = images['One photon']
mask = np.abs(reference_array) >= 0.08 * np.max(np.abs(reference_array))  # Choose threshold for comparison
for label, image in images.items():
    figure, _ = plotting.plot_image(
        image, display='both', mask=mask, normalize_magnitude=True,
        extent_cm=(-2, 2, -2, 2),
    )
    figure.suptitle(label)

In [ ]:
# Error computations (one-photon as reference_array)
reference_magnitude = np.abs(reference_array)
for label in (r'Constant $B_{1,z}$', r'Phase-modulated $B_{1,z}$'):
    candidate = images[label]
    magnitude_nrmse = (
        np.linalg.norm((np.abs(candidate) - reference_magnitude)[mask])
        / np.linalg.norm(reference_magnitude[mask])
    )
    phase_difference = np.angle(candidate[mask] * np.conj(reference_array[mask]))
    phase_rms_degrees = np.degrees(np.sqrt(np.mean(phase_difference**2)))
    print(
        f'{label}: magnitude NRMSE={magnitude_nrmse:.4f}, '
        f'phase RMS={phase_rms_degrees:.2f} degrees'
    )